In [1]:
from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd
import joblib

PROJECT_ROOT = Path(r"D:\newwwwwwww\AiBasedInstagramPrediction")

MODEL_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results" / "production"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "final_instagram_engagement_model.joblib"
FEATURE_SCHEMA_PATH = MODEL_DIR / "final_model_features.json"
METADATA_PATH = MODEL_DIR / "final_model_metadata.json"

print("=" * 70)
print("STAGE 8 — PRODUCTION PREDICTION PIPELINE")
print("=" * 70)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nModel:")
print(MODEL_PATH)

print("\nFeature schema:")
print(FEATURE_SCHEMA_PATH)

print("\nMetadata:")
print(METADATA_PATH)

assert PROJECT_ROOT.exists(), "Project root not found"
assert MODEL_PATH.exists(), "Production model not found"
assert FEATURE_SCHEMA_PATH.exists(), "Feature schema not found"
assert METADATA_PATH.exists(), "Metadata not found"

print("\n✓ All production files found")

STAGE 8 — PRODUCTION PREDICTION PIPELINE

Project root:
D:\newwwwwwww\AiBasedInstagramPrediction

Model:
D:\newwwwwwww\AiBasedInstagramPrediction\models\final_instagram_engagement_model.joblib

Feature schema:
D:\newwwwwwww\AiBasedInstagramPrediction\models\final_model_features.json

Metadata:
D:\newwwwwwww\AiBasedInstagramPrediction\models\final_model_metadata.json

✓ All production files found


In [2]:
print("=" * 70)
print("LOADING PRODUCTION MODEL")
print("=" * 70)

model = joblib.load(MODEL_PATH)

with open(FEATURE_SCHEMA_PATH, "r", encoding="utf-8") as f:
    feature_schema = json.load(f)

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("\nModel type:")
print(type(model).__name__)

print("\nModel:")
print(metadata.get("model_name", "Unknown"))

# Extract feature lists
numeric_features = feature_schema.get("numeric_features", [])
categorical_features = feature_schema.get("categorical_features", [])
all_features = feature_schema.get("all_features", [])

# Fallback if schema uses different structure
if not all_features:
    all_features = numeric_features + categorical_features

print("\nExpected feature count:", len(all_features))
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\n✓ Production model loaded")
print("✓ Feature schema loaded")
print("✓ Metadata loaded")

LOADING PRODUCTION MODEL

Model type:
Pipeline

Model:
HistGradientBoostingClassifier

Expected feature count: 56
Numeric features: 51
Categorical features: 5

✓ Production model loaded
✓ Feature schema loaded
✓ Metadata loaded


In [3]:
print("=" * 70)
print("FEATURE SCHEMA VALIDATION")
print("=" * 70)

duplicates = pd.Series(all_features)[
    pd.Series(all_features).duplicated()
].tolist()

missing_from_schema = [
    f for f in all_features
    if f not in numeric_features + categorical_features
]

print("\nTotal features:", len(all_features))
print("Numeric:", len(numeric_features))
print("Categorical:", len(categorical_features))

print("\nDuplicate features:", duplicates)
print("Unclassified features:", missing_from_schema)

assert len(all_features) == 56, (
    f"Expected 56 features, found {len(all_features)}"
)

assert len(set(all_features)) == 56, "Duplicate features detected"

print("\n✓ Exactly 56 unique production features")

FEATURE SCHEMA VALIDATION

Total features: 56
Numeric: 51
Categorical: 5

Duplicate features: []
Unclassified features: []

✓ Exactly 56 unique production features


In [4]:
print("=" * 70)
print("CREATING REAL USER INPUT")
print("=" * 70)

user_input = {
    "category": "Travel",
    "account_type": "Creator",

    "follower_count": 125000,
    "following_count": 850,
    "account_age_days": 1450,
    "verified_status": 0,
    "posting_frequency": 4.0,
    "average_historical_engagement": 0.052,
    "audience_growth_rate": 0.018,
    "account_activity_level": 0.75,
    "content_consistency": 0.70,

    "caption": (
        "Amazing sunset in Sri Lanka! 🌅✨ "
        "Such a beautiful evening by the beach. "
        "#srilanka #travel #sunset #photography #beach"
    ),

    "hashtags": "#srilanka #travel #sunset #photography #beach",

    "image_path": PROJECT_ROOT /
        "datasets" / "raw" / "instagram_data" / "img" / "insta1.jpg"
}

print("\nCategory:", user_input["category"])
print("Account type:", user_input["account_type"])
print("Followers:", user_input["follower_count"])
print("Caption:", user_input["caption"])
print("Image:", user_input["image_path"])

assert user_input["image_path"].exists(), (
    f"Test image not found: {user_input['image_path']}"
)

print("\n✓ User input created")
print("✓ Test image found")

CREATING REAL USER INPUT

Category: Travel
Account type: Creator
Followers: 125000
Caption: Amazing sunset in Sri Lanka! 🌅✨ Such a beautiful evening by the beach. #srilanka #travel #sunset #photography #beach
Image: D:\newwwwwwww\AiBasedInstagramPrediction\datasets\raw\instagram_data\img\insta1.jpg

✓ User input created
✓ Test image found


In [5]:
print("=" * 70)
print("TEXT FEATURE ENGINEERING")
print("=" * 70)

caption = user_input["caption"]
hashtags = user_input["hashtags"]

words = caption.split()

caption_length = len(caption)
word_count = len(words)
character_count = len(caption)

hashtag_count = len([
    word for word in caption.split()
    if word.startswith("#")
])

emoji_count = sum(
    1 for char in caption
    if ord(char) > 127
)

uppercase_count = sum(
    1 for char in caption
    if char.isupper()
)

exclamation_count = caption.count("!")
question_count = caption.count("?")

hashtag_length = len(hashtags)

text_features = {
    "caption_length": caption_length,
    "word_count": word_count,
    "character_count": character_count,
    "hashtag_count": hashtag_count,
    "emoji_count": emoji_count,
    "uppercase_count": uppercase_count,
    "exclamation_count": exclamation_count,
    "question_count": question_count,
    "hashtag_length": hashtag_length,
}

print("\nExtracted text features:")

for key, value in text_features.items():
    print(f"{key:30s}: {value}")

print("\n✓ Text feature extraction completed")

TEXT FEATURE ENGINEERING

Extracted text features:
caption_length                : 116
word_count                    : 18
character_count               : 116
hashtag_count                 : 5
emoji_count                   : 2
uppercase_count               : 4
exclamation_count             : 1
question_count                : 0
hashtag_length                : 45

✓ Text feature extraction completed


In [6]:
print("=" * 70)
print("IMAGE FEATURE ENGINEERING")
print("=" * 70)

from PIL import Image

image_path = user_input["image_path"]

image = Image.open(image_path)

width, height = image.size

aspect_ratio = width / height if height else 0

image_features = {
    "has_image": 1,
    "image_width": width,
    "image_height": height,
    "aspect_ratio": aspect_ratio,
}

print("\nImage:")
print(image_path)

print("\nImage dimensions:")
print("Width :", width)
print("Height:", height)

print("\nImage features:")

for key, value in image_features.items():
    print(f"{key:30s}: {value}")

print("\n✓ Image loaded successfully")

IMAGE FEATURE ENGINEERING

Image:
D:\newwwwwwww\AiBasedInstagramPrediction\datasets\raw\instagram_data\img\insta1.jpg

Image dimensions:
Width : 1080
Height: 1080

Image features:
has_image                     : 1
image_width                   : 1080
image_height                  : 1080
aspect_ratio                  : 1.0

✓ Image loaded successfully


In [7]:
print("=" * 70)
print("STAGE 8 — 56 FEATURE ASSEMBLY")
print("=" * 70)

# Start with empty dataframe
feature_row = {}

# ---------------------------------------------------------
# 1. Account / post information
# ---------------------------------------------------------

for feature in numeric_features:
    feature_row[feature] = 0.0

for feature in categorical_features:
    feature_row[feature] = ""

# ---------------------------------------------------------
# 2. User/account values
# ---------------------------------------------------------

for key in [
    "category",
    "account_type"
]:
    if key in feature_row:
        feature_row[key] = user_input[key]

for key in [
    "follower_count",
    "following_count",
    "account_age_days",
    "verified_status",
    "posting_frequency",
    "average_historical_engagement",
    "audience_growth_rate",
    "account_activity_level",
    "content_consistency",
]:
    if key in feature_row:
        feature_row[key] = user_input[key]

# ---------------------------------------------------------
# 3. Text features
# ---------------------------------------------------------

for key, value in text_features.items():
    if key in feature_row:
        feature_row[key] = value

# ---------------------------------------------------------
# 4. Image features
# ---------------------------------------------------------

for key, value in image_features.items():
    if key in feature_row:
        feature_row[key] = value

# ---------------------------------------------------------
# 5. Convert to DataFrame
# ---------------------------------------------------------

X = pd.DataFrame([feature_row])

# EXACT production order
X = X.reindex(columns=all_features)

print("\nFeature vector shape:")
print(X.shape)

print("\nMissing features:")
print(X.columns[X.isna().all()].tolist())

print("\nUnexpected features:")
print([
    c for c in X.columns
    if c not in all_features
])

assert X.shape == (1, 56), (
    f"Expected (1, 56), got {X.shape}"
)

print("\n✓ 56-feature vector assembled")

STAGE 8 — 56 FEATURE ASSEMBLY

Feature vector shape:
(1, 56)

Missing features:
[]

Unexpected features:
[]

✓ 56-feature vector assembled


In [8]:
print("=" * 70)
print("PRODUCTION DATA TYPE VALIDATION")
print("=" * 70)

# Categorical columns MUST be object/string
for column in categorical_features:
    if column in X.columns:
        X[column] = X[column].astype("object")

# Numeric columns MUST be numeric
for column in numeric_features:
    if column in X.columns:
        X[column] = pd.to_numeric(
            X[column],
            errors="coerce"
        )

# Check missing values
missing_values = X.isna().sum()

print("\nMissing values:")
print(missing_values[missing_values > 0])

assert not X.isna().any().any(), (
    "Missing values detected in production feature vector"
)

# Check duplicate columns
assert not X.columns.duplicated().any(), (
    "Duplicate feature columns detected"
)

# Verify feature order
assert list(X.columns) == list(all_features), (
    "Feature order does not match production schema"
)

print("\nCategorical dtypes:")

for column in categorical_features:
    print(
        f"{column:40s} {X[column].dtype}"
    )

print("\n✓ Data types validated")
print("✓ Feature order validated")
print("✓ No missing values")
print("✓ No duplicate features")

PRODUCTION DATA TYPE VALIDATION

Missing values:
Series([], dtype: int64)

Categorical dtypes:
category                                 object
account_type                             object
day_of_week                              object
posting_time_period                      object
media_type                               object

✓ Data types validated
✓ Feature order validated
✓ No missing values
✓ No duplicate features


In [9]:
print("=" * 70)
print("FINAL MODEL PREDICTION")
print("=" * 70)

print("\nInput shape:", X.shape)

# Prediction
prediction = model.predict(X)[0]

print("\nPredicted class:")
print(prediction)

# Probabilities
if hasattr(model, "predict_proba"):
    probabilities = model.predict_proba(X)[0]

    if hasattr(model, "classes_"):
        classes = model.classes_
    else:
        classes = ["High", "Low", "Medium"]

    probability_map = {
        str(cls): float(prob)
        for cls, prob in zip(classes, probabilities)
    }

    print("\nClass probabilities:")

    for cls, prob in probability_map.items():
        print(
            f"{cls:10s}: {prob:.4f} "
            f"({prob * 100:.2f}%)"
        )

    confidence = float(max(probabilities))

else:
    probability_map = {}
    confidence = None

print("\nPrediction:", prediction)

if confidence is not None:
    print(
        f"Confidence: {confidence:.4f} "
        f"({confidence * 100:.2f}%)"
    )

print("\n✓ Production prediction completed")

FINAL MODEL PREDICTION

Input shape: (1, 56)

Predicted class:
Low

Class probabilities:
High      : 0.0000 (0.00%)
Low       : 1.0000 (100.00%)
Medium    : 0.0000 (0.00%)

Prediction: Low
Confidence: 1.0000 (100.00%)

✓ Production prediction completed


In [10]:
print("=" * 70)
print("SAVING STAGE 8 RESULTS")
print("=" * 70)

timestamp = datetime.now().isoformat()

result = {
    "project_name": (
        "AI-Based Instagram Engagement Prediction "
        "and Content Optimization System"
    ),
    "model_name": metadata.get(
        "model_name",
        "HistGradientBoostingClassifier"
    ),
    "prediction": str(prediction),
    "confidence": confidence,
    "class_probabilities": probability_map,
    "feature_count": int(X.shape[1]),
    "model_path": str(MODEL_PATH),
    "timestamp": timestamp,
}

# JSON
json_path = RESULTS_DIR / "stage8_prediction_summary.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=4)

# CSV
csv_result = {
    "predicted_class": [str(prediction)],
    "confidence": [confidence],
}

for cls, prob in probability_map.items():
    csv_result[f"probability_{cls.lower()}"] = [prob]

csv_path = RESULTS_DIR / "stage8_prediction_result.csv"

pd.DataFrame(csv_result).to_csv(
    csv_path,
    index=False
)

print("\nJSON saved:")
print(json_path)

print("\nCSV saved:")
print(csv_path)

print("\n✓ Results saved")

SAVING STAGE 8 RESULTS

JSON saved:
D:\newwwwwwww\AiBasedInstagramPrediction\results\production\stage8_prediction_summary.json

CSV saved:
D:\newwwwwwww\AiBasedInstagramPrediction\results\production\stage8_prediction_result.csv

✓ Results saved


In [11]:
print("=" * 70)
print("STAGE 8 FINAL VERIFICATION")
print("=" * 70)

print("\nModel:")
print(type(model).__name__)

print("\nFeature vector:")
print(X.shape)

print("\nPrediction:")
print(prediction)

if confidence is not None:
    print(
        "\nConfidence:",
        f"{confidence * 100:.2f}%"
    )

print("\nOutput files:")

print(
    "✓",
    csv_path.exists(),
    csv_path
)

print(
    "✓",
    json_path.exists(),
    json_path
)

assert X.shape == (1, 56)
assert csv_path.exists()
assert json_path.exists()

print("\n" + "=" * 70)
print("STAGE 8 COMPLETED")
print("=" * 70)

print("\nNEXT:")
print("STAGE 9 — FLASK PRODUCTION API")

print("=" * 70)

STAGE 8 FINAL VERIFICATION

Model:
Pipeline

Feature vector:
(1, 56)

Prediction:
Low

Confidence: 100.00%

Output files:
✓ True D:\newwwwwwww\AiBasedInstagramPrediction\results\production\stage8_prediction_result.csv
✓ True D:\newwwwwwww\AiBasedInstagramPrediction\results\production\stage8_prediction_summary.json

STAGE 8 COMPLETED

NEXT:
STAGE 9 — FLASK PRODUCTION API
